## Objective & Tasks:

Overview: As an automotive supplier, we are interested in understanding the broader trends in the automotive industry.

Data Transformation: Transform the data into a format suitable for further analysis. Justify the choices you make during this process.

**1. Check for "Unknown" columns.**

**2. Standardize all variations of "unknown" to the unified value "Unknown".**

**3. Check which columns contain only "Unknown"**

**4. Handle French language columns.**

**5. Write to Delta with mergeSchema for schema enforcement and evolution.**

## 1. Check for "Unknown" columns:

"Unknown” is still information

Even if the value is missing, the fact that it's missing is itself meaningful.

Example:
If a station has "Unknown" as Station_Phone, that means:

- The record exists
- But the phone number was not provided

If we drop the row, we lose:

- ID
- Location
- Fuel type
- Everything else

We are discarding useful data because one column was unknown.

In [0]:
df_gold = spark.read.table(
    "hive_metastore.adfc_alternative_fuel_stations.silver"
).filter(
    "Station_Name != 'Unknown'"
)
display(df_gold)

## 2. Standardize all variations of "unknown" to the unified value "Unknown"

- No mixed casing ("unknown", "Unknown", "UNKNOWN")
- No NULLs → always "Unknown" instead
- Clean, standardized categorical fields
- Zero risk of losing records

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.adfc_alternative_fuel_stations.silver")

df_gold = df.select([
    F.when(F.col(c).isNull(), "Unknown")
     .when(F.lower(F.col(c)) == "unknown", "Unknown")
     .otherwise(F.col(c))
     .alias(c)
    if t == "string" else F.col(c)
    for c, t in df.dtypes
])

display(df_gold)

## 3. Check which columns contain only "Unknown"

- Loops through all string columns in your DataFrame.
- Checks if there are any non-"Unknown" values using .filter() with limit(1) for efficiency.
- Adds columns that are completely "Unknown" to a list.
- Drops those columns using .drop(*columns_to_drop).
- Returns a cleaned DataFrame containing only meaningful data.

Conclusion:

Reduces unnecessary columns in gold table
Keeps your analytics and reporting clean

In [0]:
from pyspark.sql import functions as F

# List to hold columns that contain only "Unknown"
columns_to_drop = []

for c, t in df_gold.dtypes:
    if t == "string":
        # Check if the column has any non-"Unknown" value
        non_unknown_count = df_gold.filter(F.col(c) != "Unknown").limit(1).count()
        if non_unknown_count == 0:
            columns_to_drop.append(c)

# Drop columns that are 100% "Unknown"
df_gold_clean = df_gold.drop(*columns_to_drop)

# Output results
print("Dropped columns with only 'Unknown':", columns_to_drop)
print("Remaining columns:", [c for c in df_gold_clean.columns])

# Optional: preview cleaned DataFrame
display(df_gold_clean)

## 4. Handle French language columns:

Justification:

Single-language gold table

- Gold layer should be clean, analytics-ready, and standardized in one language.
- Keeping French columns alongside English adds redundancy and complexity for users and BI tools.

Reduces table size and complexity

- Dropping unused French columns reduces storage, query time, and cognitive load for analysts.

Avoids confusion in reporting

- Mixing English and French column names can lead to errors in dashboards, joins, and ML pipelines.

French translations are not used

- If all downstream systems, analysts, and dashboards consume English, French columns are unnecessary.

Conclusion: Dropping all _French columns improves clarity, reduces errors, and aligns with business requirements.

In [0]:
# Identify all columns ending with '_French' in the cleaned DataFrame
french_columns = [c for c in df_gold_clean.columns if c.endswith("_French")]

# Drop all French columns
df_gold_clean = df_gold_clean.drop(*french_columns)

# Optional: show which columns were dropped
print("Dropped French columns:", french_columns)

# Preview cleaned DataFrame
df_gold_clean.printSchema()

## 5. Write to Delta with mergeSchema for schema enforcement and evolution

In [0]:
# Define the gold path and table
gold_path = "dbfs:/mnt/afdc/gold"
gold_table = "adfc_alternative_fuel_stations.gold"

# Write df_gold_clean to Delta format
df_gold_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

print(f"Gold table saved at {gold_path}")

# Create table if not exists
spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_table} USING DELTA LOCATION '{gold_path}'")

# Verify the table
display(spark.sql(f"SELECT * FROM {gold_table} LIMIT 10"))

In [0]:
df = spark.read.table("hive_metastore.adfc_alternative_fuel_stations.gold").display()